# The TW Project

## Stage 1: Finetune a YOLO26s on LLVIP dataset

### Data preparation

In [ ]:
"""Download the dataset"""

import os

def download_dataset():
    import gdown

    url = "https://drive.google.com/uc?id=1VTlT3Y7e1h-Zsne4zahjx5q0TK2ClMVv"
    gdown.download(
        url=url,
        output="datasets.zip",
    )

if (not os.path.exists("datasets.zip")):
    print("Dataset not downloaded, now downloading")
    download_dataset()
else:
    import hashlib

    with open("datasets.zip", "rb") as f:
        file_hash = hashlib.md5()
        while chunk := f.read(8192):
            file_hash.update(chunk)
    if file_hash.hexdigest() != "e64affb4b0b50e1772ff6f67da873bf6":
        print("Download dataset again due to hash mismatch")
        download_dataset()

if (not os.path.exists("datasets")):
    import zipfile
    
    with zipfile.ZipFile("datasets.zip", "r") as zip_ref:
        zip_ref.extractall("datasets")

In [ ]:
"""Convert VOC format to YOLO format."""

from pathlib import Path
import xml.etree.ElementTree as ET
from tqdm.notebook import tqdm

base_path = Path("datasets/LLVIP")
TRAIN_COUNT = 10684
VAL_COUNT = 1341
TEST_COUNT = 3463

def convert_label(xml_path, lb_path):
    """Converts XML annotations from VOC format to YOLO format by extracting bounding boxes and class IDs."""
    def convert_box(size, box):
        dw, dh = 1.0 / size[0], 1.0 / size[1]
        x, y, w, h = (box[0] + box[1]) / 2.0 - 1, (box[2] + box[3]) / 2.0 - 1, box[1] - box[0], box[3] - box[2]
        return x * dw, y * dh, w * dw, h * dh

    if not xml_path.exists():
        return

    with open(xml_path, "r", encoding="utf-8") as in_file, open(lb_path, "w", encoding="utf-8") as out_file:
        tree = ET.parse(in_file)
        root = tree.getroot()
        size = root.find("size")
        w = int(size.find("width").text)
        h = int(size.find("height").text)

        for obj in root.iter("object"):
            cls = obj.find("name").text
            if cls == "person":
                xmlbox = obj.find("bndbox")
                bb = convert_box((w, h), [float(xmlbox.find(x).text) for x in ("xmin", "xmax", "ymin", "ymax")])
                out_file.write(f"0 {' '.join(str(a) for a in bb)}\n")

# Create directories
for folder in ["images", "labels"]:
    for split in ["train", "val", "test"]:
        (base_path / folder / split).mkdir(parents=True, exist_ok=True)

# Process train/val
train_images = sorted(list((base_path / "infrared" / "train").glob("*.jpg")))
assert len(train_images) == TRAIN_COUNT + VAL_COUNT

for i, img_path in enumerate(tqdm(train_images, desc="Processing train/val")):
    split = "train" if i < TRAIN_COUNT else "val"
    new_img_path = base_path / "images" / split / img_path.name
    lb_path = base_path / "labels" / split / img_path.with_suffix(".txt").name
    xml_path = base_path / "Annotations" / img_path.with_suffix(".xml").name
    
    img_path.rename(new_img_path)
    convert_label(xml_path, lb_path)

# Process test
test_images = sorted(list((base_path / "infrared" / "test").glob("*.jpg")))
assert len(test_images) == TEST_COUNT

for img_path in tqdm(test_images, desc="Processing test"):
    split = "test"
    new_img_path = base_path / "images" / split / img_path.name
    lb_path = base_path / "labels" / split / img_path.with_suffix(".txt").name
    xml_path = base_path / "Annotations" / img_path.with_suffix(".xml").name
    
    img_path.rename(new_img_path)
    convert_label(xml_path, lb_path)


In [8]:
"""Create dataset YAML file"""

yaml_path = base_path / "llvip.yaml"
yaml_content = f"""path: {base_path}
train: {base_path / 'images' / 'train'}
val: {base_path / 'images' / 'val'}
test: {base_path / 'images' / 'test'}
nc: 1
names: ['person']
"""

yaml_path.write_text(yaml_content, encoding="utf-8")

144

### Tuning and training

In [ ]:
from pathlib import Path
from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionTrainer


class CustomSaveTrainer(DetectionTrainer):
    """Trainer that saves the best model based on recall instead of default fitness."""

    def validate(self):
        """Override fitness to use recall for best model selection."""
        metrics, fitness = super().validate()
        if metrics:
            fitness = metrics.get("metrics/recall(B)", fitness)
            if self.best_fitness is None or fitness > self.best_fitness:
                self.best_fitness = fitness
        return metrics, fitness


yaml_path = Path("datasets/LLVIP/llvip.yaml").resolve()

In [ ]:
"""Train stage 1"""

from ultralytics import YOLO

model = YOLO("yolov26s.pt")
model.train(
    data=yaml_path,
    epochs=20,
    imgsz=640,
    batch=16,
    name="llvip_yolov26n_stage_1",
    patience=10,
    cache=True,
    profile=True,
    freeze=10,
    # Augmentation
    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.2,
    degrees=0.0,
    translate=0.1,
    scale=0.4,
    shear=0.0,
    perspective=0.0,
    flipud=0.0,
    fliplr=0.5,
    mosaic=0.5,
    mixup=0.0,
    erasing=0.2,
)

In [ ]:
"""Train stage 2"""

from ultralytics import YOLO

model = YOLO("llvip_yolov26n_stage_1.pt") # change
model.train(
    data=yaml_path,
    epochs=70,
    imgsz=640,
    batch=16,
    lr0=1e-4,
    name="llvip_yolov26n_stage_2",
    patience=10,
    cache=True,
    profile=True,
    # Augmentation
    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.4,
    degrees=15.0,
    translate=0.2,
    scale=0.6,
    shear=5.0,
    perspective=0.0002,
    flipud=0.0,
    fliplr=0.5,
    mosaic=0.2,
    mixup=0.0,
    erasing=0.4,
)